# Amazon Week 5 V1 — One-Click Feature Dataset

**How to use:** Runtime → Run all → upload the 4 ZIP files when prompted.


In [ ]:
# ============================================================
# WEEK 5 V1 — ONE-CLICK FEATURE DATASET
# Amazon Search / Learning-to-Rank Project
# Inputs:
#   1) amazon_week1_v4_processed_data*.zip
#   2) amazon_week2_baseline_ranking_results*.zip
#   3) amazon_week4_v1_sbert_faiss_results*.zip
#   4) amazon_week4_v2_clip_multimodal_results*.zip
#
# Output:
#   amazon_week5_v1_feature_dataset.zip
# ============================================================

!pip -q install sentence-transformers transformers accelerate pyarrow

import os, re, json, shutil, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPTokenizerFast

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("/content/week5_v1")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

print("STEP 1/8 — Upload the 4 ZIP files")
uploaded = files.upload()
zip_names = list(uploaded.keys())

def pick(patterns):
    for name in zip_names:
        low = name.lower()
        if all(p.lower() in low for p in patterns):
            return name
    return None

w1_zip = pick(["week1", "processed"])
w2_zip = pick(["week2", "baseline"])
w4v1_zip = pick(["week4", "v1", "sbert"])
w4v2_zip = pick(["week4", "v2", "clip"])

required = {
    "Week1 processed data": w1_zip,
    "Week2 baseline results": w2_zip,
    "Week4 V1 SBERT": w4v1_zip,
    "Week4 V2 CLIP": w4v2_zip,
}
missing = [k for k, v in required.items() if v is None]
if missing:
    raise ValueError("Missing ZIP(s): " + ", ".join(missing))

print("Detected:")
for k,v in required.items():
    print(f"  {k}: {v}")

def extract_zip(name, subdir):
    out = ROOT / subdir
    out.mkdir(exist_ok=True)
    with zipfile.ZipFile(name, "r") as z:
        z.extractall(out)
    return out

W1 = extract_zip(w1_zip, "week1")
W2 = extract_zip(w2_zip, "week2")
W4V1 = extract_zip(w4v1_zip, "week4_v1")
W4V2 = extract_zip(w4v2_zip, "week4_v2")

def must_find(folder, filename):
    hits = list(folder.rglob(filename))
    if not hits:
        raise FileNotFoundError(f"{filename} not found under {folder}")
    return hits[0]

print("\nSTEP 2/8 — Load Week 1 / Week 2 data")

pairs = pd.read_parquet(must_find(W1, "query_product_relevance.parquet"))
catalog = pd.read_parquet(must_find(W1, "cleaned_products_10k.parquet"))
rank = pd.read_parquet(must_find(W2, "ranking_features.parquet"))
train_q = pd.read_csv(must_find(W2, "fixed_train_queries.csv"))
test_q = pd.read_csv(must_find(W2, "fixed_test_queries.csv"))

for df in [pairs, catalog, rank]:
    if "product_id" in df.columns:
        df["product_id"] = df["product_id"].astype(str)
    if "query_id" in df.columns:
        df["query_id"] = df["query_id"].astype(str)

train_ids = set(train_q["query_id"].astype(str))
test_ids = set(test_q["query_id"].astype(str))

print(f"Week1 judged pairs: {len(pairs):,}")
print(f"Queries: {pairs['query_id'].nunique():,}")
print(f"Catalog products: {len(catalog):,}")
print(f"Week2 ranking rows: {len(rank):,}")
print(f"Train queries: {len(train_ids):,} | Test queries: {len(test_ids):,}")

base = rank.copy()

meta_cols = [
    "query_id","query","product_id","title","description","category","brand",
    "price","average_rating","review_count","relevance_label",
    "price_missing","description_missing","image_missing","brand_missing"
]
meta_cols = [c for c in meta_cols if c in pairs.columns]
meta = pairs[meta_cols].drop_duplicates(["query_id","product_id"])

for c in meta.columns:
    if c in ["query_id","product_id"]:
        continue
    if c not in base.columns:
        base = base.merge(
            meta[["query_id","product_id",c]],
            on=["query_id","product_id"],
            how="left"
        )

if "relevance_label" not in base.columns:
    base = base.merge(
        pairs[["query_id","product_id","relevance_label"]],
        on=["query_id","product_id"],
        how="left"
    )

base["split"] = np.where(
    base["query_id"].isin(train_ids), "train",
    np.where(base["query_id"].isin(test_ids), "test", "unknown")
)
if (base["split"] == "unknown").any():
    raise ValueError("Some rows are not covered by the fixed Week2 train/test split.")

def norm_text(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"[^a-z0-9\s]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

base["query"] = base["query"].fillna("").astype(str)
base["title"] = base["title"].fillna("").astype(str)

print("\nSTEP 3/8 — Add SBERT similarity using Week4 V1 product embeddings")

sbert_product_emb = np.load(
    must_find(W4V1, "week4_sbert_product_embeddings.npy")
).astype(np.float32)

if len(sbert_product_emb) != len(catalog):
    raise ValueError(
        f"SBERT embedding/catalog mismatch: {len(sbert_product_emb)} vs {len(catalog)}"
    )

catalog = catalog.reset_index(drop=True)
catalog["catalog_index"] = np.arange(len(catalog))
product_to_catalog_idx = dict(
    zip(catalog["product_id"].astype(str), catalog["catalog_index"].astype(int))
)

sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

query_table = (
    base[["query_id","query"]]
    .drop_duplicates("query_id")
    .sort_values("query_id")
    .reset_index(drop=True)
)

sbert_q = sbert_model.encode(
    query_table["query"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype(np.float32)

qid_to_sbert_row = dict(zip(query_table["query_id"], range(len(query_table))))
q_rows = base["query_id"].map(qid_to_sbert_row).to_numpy()
p_rows = base["product_id"].map(product_to_catalog_idx)

valid_sbert = p_rows.notna().to_numpy()
sbert_scores = np.zeros(len(base), dtype=np.float32)
p_rows_np = p_rows.fillna(0).astype(int).to_numpy()

sbert_scores[valid_sbert] = np.sum(
    sbert_q[q_rows[valid_sbert]] *
    sbert_product_emb[p_rows_np[valid_sbert]],
    axis=1
)

base["sbert_similarity"] = sbert_scores
base["sbert_available"] = valid_sbert.astype(np.int8)
print(f"SBERT coverage: {base['sbert_available'].mean()*100:.2f}%")

print("\nSTEP 4/8 — Add CLIP text-image similarity using Week4 V2 image embeddings")

clip_img_emb = np.load(
    must_find(W4V2, "week4_clip_image_embeddings.npy")
).astype(np.float32)
clip_map = pd.read_csv(
    must_find(W4V2, "week4_clip_embedding_map.csv")
)
clip_map["product_id"] = clip_map["product_id"].astype(str)

if len(clip_img_emb) != len(clip_map):
    raise ValueError(
        f"CLIP embedding/map mismatch: {len(clip_img_emb)} vs {len(clip_map)}"
    )

product_to_clip_row = dict(
    zip(clip_map["product_id"], clip_map["clip_row"].astype(int))
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("CLIP device:", device)

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)
clip_tokenizer = CLIPTokenizerFast.from_pretrained(
    "openai/clip-vit-base-patch32"
)
clip_model.eval()

def encode_clip_text(texts, batch_size=128):
    outputs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            tok = clip_tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=77,
                return_tensors="pt"
            ).to(device)
            feats = clip_model.get_text_features(**tok)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            outputs.append(feats.cpu().numpy().astype(np.float32))
    return np.vstack(outputs)

clip_q = encode_clip_text(query_table["query"].tolist())
qid_to_clip_row = dict(zip(query_table["query_id"], range(len(query_table))))

clip_q_rows = base["query_id"].map(qid_to_clip_row).to_numpy()
clip_p_rows = base["product_id"].map(product_to_clip_row)
valid_clip = clip_p_rows.notna().to_numpy()
clip_p_rows_np = clip_p_rows.fillna(0).astype(int).to_numpy()

clip_scores = np.zeros(len(base), dtype=np.float32)
clip_scores[valid_clip] = np.sum(
    clip_q[clip_q_rows[valid_clip]] *
    clip_img_emb[clip_p_rows_np[valid_clip]],
    axis=1
)

base["clip_similarity"] = clip_scores
base["clip_available"] = valid_clip.astype(np.int8)
print(f"CLIP image coverage on judged pairs: {base['clip_available'].mean()*100:.2f}%")

print("\nSTEP 5/8 — Add Week5 interaction / metadata features")

for col, default in [
    ("price", np.nan),
    ("average_rating", np.nan),
    ("review_count", 0),
]:
    if col not in base.columns:
        base[col] = default

base["price"] = pd.to_numeric(base["price"], errors="coerce")
base["average_rating"] = pd.to_numeric(base["average_rating"], errors="coerce")
base["review_count"] = pd.to_numeric(
    base["review_count"], errors="coerce"
).fillna(0)

base["log_price"] = np.log1p(base["price"].fillna(0).clip(lower=0))
base["log_review_count_w5"] = np.log1p(base["review_count"].clip(lower=0))
base["rating_missing_w5"] = base["average_rating"].isna().astype(np.int8)
base["price_missing_w5"] = base["price"].isna().astype(np.int8)

q_norm = base["query"].map(norm_text)
t_norm = base["title"].map(norm_text)

base["exact_title_match"] = (q_norm == t_norm).astype(np.int8)
base["query_in_title"] = [
    int(bool(q) and q in t) for q,t in zip(q_norm, t_norm)
]

def token_overlap(a, b):
    A, B = set(a.split()), set(b.split())
    if not A:
        return 0.0
    return len(A & B) / len(A)

base["query_title_token_coverage"] = [
    token_overlap(q,t) for q,t in zip(q_norm,t_norm)
]

if "brand" in base.columns:
    b_norm = base["brand"].fillna("").astype(str).map(norm_text)
    base["brand_in_query"] = [
        int(bool(b) and b in q) for q,b in zip(q_norm,b_norm)
    ]
else:
    base["brand_in_query"] = 0

if "category" in base.columns:
    c_norm = base["category"].fillna("").astype(str).map(norm_text)
    base["category_token_match"] = [
        token_overlap(q,c) for q,c in zip(q_norm,c_norm)
    ]
else:
    base["category_token_match"] = 0.0

score_candidates = [
    "bm25_score",
    "tfidf_similarity",
    "sbert_similarity",
    "clip_similarity"
]

def minmax_group(s):
    lo = s.min()
    hi = s.max()
    if pd.isna(lo) or pd.isna(hi) or hi <= lo:
        return pd.Series(np.zeros(len(s), dtype=np.float32), index=s.index)
    return ((s - lo) / (hi - lo)).astype(np.float32)

for c in score_candidates:
    if c in base.columns:
        base[c + "_qnorm"] = (
            base.groupby("query_id", group_keys=False)[c]
                .apply(minmax_group)
                .astype(np.float32)
        )

if "query_len" not in base.columns:
    base["query_len"] = q_norm.str.split().str.len().astype(int)
if "title_len" not in base.columns:
    base["title_len"] = t_norm.str.split().str.len().astype(int)

base["relevance_label"] = pd.to_numeric(
    base["relevance_label"], errors="coerce"
).astype("Int64")

print("\nSTEP 6/8 — Validation")

assert base[["query_id","product_id"]].duplicated().sum() == 0
assert base["relevance_label"].notna().all()
assert set(base["split"].unique()) <= {"train","test"}

print("Rows:", len(base))
print("Queries:", base["query_id"].nunique())
print("Products in judged set:", base["product_id"].nunique())
print("Train rows:", (base["split"]=="train").sum())
print("Test rows:", (base["split"]=="test").sum())
print("\nLabel distribution:")
print(base["relevance_label"].value_counts().sort_index())

preferred_features = [
    "bm25_score","tfidf_similarity",
    "price","average_rating","log_review_count",
    "query_len","title_len","description_len",
    "query_title_overlap","query_description_overlap",
    "sbert_similarity","clip_similarity",
    "sbert_available","clip_available",
    "log_price","log_review_count_w5",
    "rating_missing_w5","price_missing_w5",
    "exact_title_match","query_in_title",
    "query_title_token_coverage",
    "brand_in_query","category_token_match",
    "bm25_score_qnorm","tfidf_similarity_qnorm",
    "sbert_similarity_qnorm","clip_similarity_qnorm"
]
feature_cols = [c for c in preferred_features if c in base.columns]
feature_cols = [
    c for c in feature_cols if pd.api.types.is_numeric_dtype(base[c])
]

schema = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [str(base[c].dtype) for c in feature_cols],
    "missing_count": [int(base[c].isna().sum()) for c in feature_cols],
    "missing_pct": [float(base[c].isna().mean()*100) for c in feature_cols],
})

summary = pd.DataFrame([
    ["rows", len(base)],
    ["unique_queries", base["query_id"].nunique()],
    ["unique_products", base["product_id"].nunique()],
    ["train_queries", len(train_ids)],
    ["test_queries", len(test_ids)],
    ["train_rows", int((base["split"]=="train").sum())],
    ["test_rows", int((base["split"]=="test").sum())],
    ["sbert_coverage_pct", round(base["sbert_available"].mean()*100, 4)],
    ["clip_coverage_pct", round(base["clip_available"].mean()*100, 4)],
    ["num_model_features", len(feature_cols)],
], columns=["metric","value"])

print("\nModel features:")
for i,c in enumerate(feature_cols,1):
    print(f"{i:02d}. {c}")

print("\nSTEP 7/8 — Save Week5 V1 outputs")

OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

base.to_parquet(OUT / "week5_feature_dataset.parquet", index=False)
base.to_csv(OUT / "week5_feature_dataset.csv", index=False)
schema.to_csv(OUT / "week5_feature_schema.csv", index=False)
summary.to_csv(OUT / "week5_feature_summary.csv", index=False)
train_q.to_csv(OUT / "week5_train_queries.csv", index=False)
test_q.to_csv(OUT / "week5_test_queries.csv", index=False)

with open(OUT / "week5_feature_list.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

readme = f"""
AMAZON SEARCH / LTR — WEEK 5 V1 FEATURE DATASET
================================================

Purpose
-------
Build the query-product feature table for LambdaMART.

Rows: {len(base):,}
Queries: {base['query_id'].nunique():,}
Train queries: {len(train_ids)}
Test queries: {len(test_ids)}
Model features: {len(feature_cols)}

Target
------
relevance_label from Week1 V4 Amazon relationship-graph labels.

Feature groups
--------------
1. Lexical: BM25, TF-IDF, Week2 overlap features
2. Semantic: SBERT cosine similarity
3. Multimodal: CLIP query-to-image cosine similarity
4. Product metadata: price, rating, review count
5. Interaction: exact/title/brand/category matching

Split
-----
The exact Week2 fixed query-level split is preserved:
800 train queries / 200 test queries.

Important metric note
---------------------
The uploaded Week2 pointwise NDCG@10 baseline is 0.922169.
A 12% relative improvement over this would require NDCG > 1.0,
which is mathematically impossible.

Recommended Week5 acceptance wording:
- LambdaMART should outperform the Week2 XGBoost pointwise baseline; AND
- LambdaMART should be at least 12% above the Week2 BM25 baseline on NDCG@10.

Week3 LLM intent features are not included in this V1 because no Week3 artifact
was provided. They can be added later as an optional ablation feature group.
"""
with open(OUT / "README_WEEK5_V1.txt", "w") as f:
    f.write(readme.strip() + "\n")

zip_out = Path("/content/amazon_week5_v1_feature_dataset.zip")
if zip_out.exists():
    zip_out.unlink()

with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.name)

print("\nSTEP 8/8 — DONE")
print(summary.to_string(index=False))
print(f"\nCreated: {zip_out}")
files.download(str(zip_out))
